In [ ]:
import os
import pandas as pd

metrics = ["ssim_index", "buffer", "cum_rebuffer", "rtt", "delivery_rate", "size"]

def process_trace_dir(directory: str) -> pd.DataFrame:
    results = []

    if not os.path.exists(directory):
        print(f"Directory does not exist: {directory}")
        return pd.DataFrame()

    files_found = False
    for filename in os.listdir(directory):
        if filename.lower().endswith(".csv"):
            files_found = True
            full_path = os.path.join(directory, filename)
            try:
                df = pd.read_csv(full_path)
                means = {
                    metric: (df[metric].iloc[-1]
                             if metric == "cum_rebuffer"
                             else df[metric].mean())
                    for metric in metrics
                    if metric in df.columns
                }

                abr_name = os.path.splitext(filename)[0]
                means["abr"] = abr_name

                folder_parts = directory.lower().split(os.sep)
                if "mahimahi" in folder_parts:
                    means["emulator"] = "mahimahi"
                elif "cellreplay" in folder_parts:
                    means["emulator"] = "cellreplay"
                else:
                    means["emulator"] = "unknown"


                means["trace_path"] = directory
                means["file"] = filename
                results.append(means)
            except Exception as e:
                print(f"Error reading {filename}: {e}")

    if not files_found:
        print(f"No CSV files found in {directory}")

    return pd.DataFrame(results).rename(columns={
        "ssim_index": "Mean SSIM",
        "buffer": "Mean Buffer (s)",
        "cum_rebuffer": "Mean Rebuffer (s)",
        "rtt": "Mean RTT (µs)",
        "delivery_rate": "Mean Delivery Rate (bytes/s)",
        "size": "Mean Chunk Size (bytes)"
    })


In [ ]:
tmobile_cellreplay_driving_df = process_trace_dir(
    r"" # Path for CSV files
)
tmobile_cellreplay_driving_df.head()


In [ ]:
tmobile_cellreplay_weak_df = process_trace_dir(
    r"" # Path for CSV files
)
tmobile_cellreplay_weak_df.head()


In [ ]:
tmobile_cellreplay_weak_df = process_trace_dir(
    r"" # Path for CSV files
)
tmobile_cellreplay_weak_df.head()

In [ ]:
verizon_cellreplay_crowded_df = process_trace_dir(
    r"" # Path for CSV files
)
verizon_cellreplay_crowded_df.head()


In [ ]:
verizon_cellreplay_walking_df = process_trace_dir(
    r"" # Path for CSV files
)
verizon_cellreplay_walking_df.head()


In [ ]:
verizon_cellreplay_walking_df = process_trace_dir(
    r"" # Path for CSV files
)
verizon_cellreplay_walking_df.head()

In [ ]:
tmobile_mahimahi_driving_df = process_trace_dir(
    r"" # Path for CSV files
)
tmobile_mahimahi_driving_df.head()


In [ ]:
tmobile_mahimahi_weak_df = process_trace_dir(
    r"" # Path for CSV files
)
tmobile_mahimahi_weak_df.head()


In [ ]:
tmobile_mahimahi_weak_df = process_trace_dir(
    r"" # Path for CSV files
)
tmobile_mahimahi_weak_df.head()


In [ ]:
verizon_mahimahi_crowded_df = process_trace_dir(
    r"" # Path for CSV files
)
verizon_mahimahi_crowded_df.head()


In [ ]:
verizon_mahimahi_walking_df = process_trace_dir(
    r"" # Path for CSV files
)
verizon_mahimahi_walking_df.head()


In [ ]:
verizon_mahimahi_walking_df = process_trace_dir(
    r"" # Path for CSV files
)
verizon_mahimahi_walking_df.head()

In [ ]:
trace_dirs = [
r"" # Path for CSV files
]

dfs = [process_trace_dir(path) for path in trace_dirs]
combined_df = pd.concat(dfs, ignore_index=True)


In [ ]:
abr_avg_df = combined_df.groupby(['abr', 'emulator']).mean(numeric_only=True).reset_index()
abr_avg_df = abr_avg_df.round(8)

abr_avg_df['SSIM Rank'] = abr_avg_df.groupby('emulator')["Mean SSIM"].rank(ascending=False)
abr_avg_df['Rebuffer Rank'] = abr_avg_df.groupby('emulator')["Mean Rebuffer (s)"].rank(ascending=True)
abr_avg_df["Buffer Rank"] = abr_avg_df.groupby("emulator")["Mean Buffer (s)"].rank(ascending=True)

abr_avg_df["RTT Rank"] = abr_avg_df.groupby("emulator")["Mean RTT (µs)"].rank(ascending=True)
abr_avg_df["Delivery Rate Rank"] = abr_avg_df.groupby("emulator")["Mean Delivery Rate (bytes/s)"].rank(ascending=False)
abr_avg_df["Chunk Size Rank"] = abr_avg_df.groupby("emulator")["Mean Chunk Size (bytes)"].rank(ascending=False)


In [ ]:
abr_mahimahi = abr_avg_df[abr_avg_df['emulator'] == 'mahimahi']
abr_cellreplay = abr_avg_df[abr_avg_df['emulator'] == 'cellreplay']

abr_mahimahi


In [ ]:
abr_cellreplay


In [ ]:
PLAY_TIME = 600

abr_avg_df["BufRatio (%)"] = (
    abr_avg_df["Mean Rebuffer (s)"] / (abr_avg_df["Mean Rebuffer (s)"] + PLAY_TIME)
) * 100

abr_mahimahi_buf = abr_avg_df[abr_avg_df["emulator"] == "mahimahi"][["abr", "BufRatio (%)"]].round(5)
abr_cellreplay_buf = abr_avg_df[abr_avg_df["emulator"] == "cellreplay"][["abr", "BufRatio (%)"]].round(5)

print("Mahimahi BufRatio (%):")
print(abr_mahimahi_buf.to_string(index=False))

print("\nCellReplay BufRatio (%):")
print(abr_cellreplay_buf.to_string(index=False))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from cycler import cycler
import os

PLAY_TIME = 600 

abr_avg_df["BufRatio (%)"] = (
    abr_avg_df["Mean Rebuffer (s)"] /
    (abr_avg_df["Mean Rebuffer (s)"] + PLAY_TIME)
) * 100

abr_mahimahi_buf  = (
    abr_avg_df[abr_avg_df["emulator"] == "mahimahi"]
    .copy()
    .round(5)
)
abr_cellreplay_buf = (
    abr_avg_df[abr_avg_df["emulator"] == "cellreplay"]
    .copy()
    .round(5)
)

OUTPUT_PDF_DIR = (
    r"C:\Users\erine\Thesis\ABR-evaluation---thesis"
    r"\mahimahi-cellreplay\graphs-pdf"
)
os.makedirs(OUTPUT_PDF_DIR, exist_ok=True)

SET1_6 = ['#e41a1c', '#377eb8', '#4daf4a',
          '#984ea3', '#ff7f00', '#a65628']

SSIM_BUFR_RC = {
    'figure.figsize': (5, 3),
    'font.size': 9,
    'axes.labelsize': 9,
    'axes.titlesize': 10,
    'legend.fontsize': 8,
    'lines.linewidth': 1.5,
    'axes.prop_cycle': cycler('color', SET1_6),
    'font.family': 'sans-serif',
}

def plot_ssim_vs_bufratio(df, emulator_name, fname_stub):
    """Scatter-plot Mean SSIM versus BufRatio (%) for one emulator."""
    with plt.rc_context(SSIM_BUFR_RC):
        fig, ax = plt.subplots()
        fig.set_size_inches(5, 3)

        for abr in df["abr"].unique():
            row = df[df["abr"] == abr].iloc[0]
            ax.scatter(
                row["BufRatio (%)"],
                row["Mean SSIM"],
                s=80,
                marker="x",
                label=abr,
            )

        ax.set_xlabel("BufRatio (%)")
        ax.set_ylabel("SSIM")

        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        ax.legend(
            loc="lower right",
            frameon=False
        )

        ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
        ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
        ax.grid(which="major", linestyle="dashdot",
                linewidth=0.4, color="#AEAEAE")
        ax.grid(which="minor", linestyle="dotted",
                linewidth=0.2, color="#AEAEAE")

        plt.tight_layout()
        pdf_path = os.path.join(OUTPUT_PDF_DIR, f"{fname_stub}.pdf")
        fig.savefig(pdf_path, format="pdf", bbox_inches="tight")
        print(f"Saved → {pdf_path}")
        plt.show()

plot_ssim_vs_bufratio(abr_mahimahi_buf,   "Mahimahi 5G",   "ssim_bufratio_mahimahi")
plot_ssim_vs_bufratio(abr_cellreplay_buf, "CellReplay 5G", "ssim_bufratio_cellreplay")


In [ ]:
import matplotlib.pyplot as plt

OUTPUT_PDF_DIR = r"" # Path for output directory
os.makedirs(OUTPUT_PDF_DIR, exist_ok=True)

abr_mahimahi["abr"] = abr_mahimahi["abr"].replace({"Linear BBA": "Linear"})
abr_cellreplay["abr"] = abr_cellreplay["abr"].replace({"Linear BBA": "Linear"})

buffer_mahimahi   = abr_mahimahi.sort_values("Mean Buffer (s)")
buffer_cellreplay = abr_cellreplay.sort_values("Mean Buffer (s)")

min_buffer_mah   = buffer_mahimahi["Mean Buffer (s)"].min()
max_buffer_mah   = buffer_mahimahi["Mean Buffer (s)"].max()
min_buffer_cell  = buffer_cellreplay["Mean Buffer (s)"].min()
max_buffer_cell  = buffer_cellreplay["Mean Buffer (s)"].max()

margin = 0.005

fig, ax = plt.subplots()
fig.set_size_inches(5, 3)

ax.bar(buffer_mahimahi["abr"],
       buffer_mahimahi["Mean Buffer (s)"],
       color='cornflowerblue')

ax.set_xlabel("ABR Algorithm")
ax.set_ylabel("Buffer Time (s)")
ax.set_ylim(min_buffer_mah - margin, max_buffer_mah + margin)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_PDF_DIR, "buffer_barchart_mahimahi5G.pdf"), bbox_inches="tight")
plt.show()

fig, ax = plt.subplots()
fig.set_size_inches(5, 3)

ax.bar(buffer_cellreplay["abr"],
       buffer_cellreplay["Mean Buffer (s)"],
       color='mediumseagreen')

ax.set_xlabel("ABR Algorithm")
ax.set_ylabel("Buffer Time (s)")
ax.set_ylim(min_buffer_cell - margin, max_buffer_cell + margin)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_PDF_DIR, "buffer_barchart_cellreplay5G.pdf"), bbox_inches="tight")
plt.show()
